# Stage 02 - Vector Embeddings: Visualization & Cosine Similarity Mathematics
### Lecture 20: 2D Dimensionality Reduction (PCA) and Angular Proximity

This notebook provides visual and mathematical proof of vector space topology:
1. Implementing cosine similarity from pure mathematical definitions:
   $$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$
2. Dimensionality reduction using Principal Component Analysis (PCA) to project 384D to 2D.
3. Visualizing semantic clusters (Technology vs Finance vs Biology).


In [1]:
# Step 1: Mathematical Cosine Similarity Implementation
import numpy as np

def compute_cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Compute cosine similarity using direct dot product and Euclidean norm."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return float(dot_product / (norm_a * norm_b))

# Verification vectors
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([2.0, 4.0, 6.0])  # Collinear vector (angle 0)
v3 = np.array([-1.0, -2.0, -3.0]) # Opposite vector (angle 180)

sim_collinear = compute_cosine_similarity(v1, v2)
sim_opposite = compute_cosine_similarity(v1, v3)

print(f"[MATH] Collinear Vectors Similarity: {sim_collinear:.4f}")
print(f"[MATH] Opposite Vectors Similarity: {sim_opposite:.4f}")

# Assertions
assert np.isclose(sim_collinear, 1.0), "Collinear vectors must yield cosine similarity of 1.0."
assert np.isclose(sim_opposite, -1.0), "Opposite vectors must yield cosine similarity of -1.0."


[MATH] Collinear Vectors Similarity: 1.0000
[MATH] Opposite Vectors Similarity: -1.0000


In [2]:
# Step 2: Multi-Domain Semantic Clustering
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

corpus = [
    # Domain 1: AI & Computing
    "Deep neural networks require GPU hardware acceleration.",
    "Transformers utilize self-attention mechanisms for language understanding.",
    "Backpropagation calculates gradients for model weight updates.",
    # Domain 2: Finance & Markets
    "Central banks manage inflation using interest rate adjustments.",
    "Stock market indices experienced high volatility during trading.",
    "Corporate balance sheets report assets liabilities and equity.",
    # Domain 3: Biology & Medicine
    "Mitochondria generate cellular energy in the form of ATP.",
    "DNA transcription produces messenger RNA molecules.",
    "Antibodies bind to antigens to neutralize pathogenic infections."
]

labels = ["AI", "AI", "AI", "Finance", "Finance", "Finance", "Biology", "Biology", "Biology"]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings_matrix = model.encode(corpus)

# Project high-dimensional vectors to 2D using PCA
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embeddings_matrix)

print(f"[INFO] High-dimensional shape: {embeddings_matrix.shape}")
print(f"[INFO] 2D projection shape: {coords_2d.shape}")
print(f"[INFO] Explained variance ratio: {np.sum(pca.explained_variance_ratio_):.4f}")

# Assertions
assert coords_2d.shape == (9, 2), "PCA output must be (9, 2)."


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] High-dimensional shape: (9, 384)
[INFO] 2D projection shape: (9, 2)
[INFO] Explained variance ratio: 0.3480


In [3]:
# Step 3: Visualization and Cluster Separation Verification
# Compute intra-cluster vs inter-cluster similarity
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings_matrix)

# Average similarity within AI domain (items 0, 1, 2)
intra_ai = np.mean([sim_matrix[0, 1], sim_matrix[0, 2], sim_matrix[1, 2]])
# Average similarity between AI and Biology (items 0 vs 6, 7, 8)
inter_ai_bio = np.mean(sim_matrix[0:3, 6:9])

print(f"[METRICS] Intra-cluster AI Similarity: {intra_ai:.4f}")
print(f"[METRICS] Inter-cluster AI-Biology Similarity: {inter_ai_bio:.4f}")

# Assertions
assert intra_ai > inter_ai_bio, "Intra-cluster similarity must strictly exceed inter-cluster similarity."
assert intra_ai > 0.15 and inter_ai_bio < 0.10, "Intra-cluster similarity must exceed inter-cluster baseline."
print("[SUCCESS] Vector space topology, cosine mathematics, and clustering verified.")


[METRICS] Intra-cluster AI Similarity: 0.2344
[METRICS] Inter-cluster AI-Biology Similarity: 0.0468
[SUCCESS] Vector space topology, cosine mathematics, and clustering verified.
